# DA4 Assignment 2 — Regression Analysis
**Question:** To what extent does economic activity cause CO2 emissions?

We estimate 6 models using ln(CO2 pc) as the dependent variable and ln(GDP pc) as the
main regressor, then add urbanization as a confounder to models 1, 4, and 6.

In [1]:
import pandas as pd
import numpy as np
import pyfixest as pf
import warnings
warnings.filterwarnings('ignore')

OUT = '../output'

In [2]:
df = pd.read_csv('../data/clean/wdi_clean.csv')
df = df.rename(columns={'Country Code': 'country', 'Country Name': 'country_name'})
print(f'{df.country.nunique()} countries, {df.year.min()}-{df.year.max()}')

190 countries, 1992-2023


## 1. Cross-section OLS

In [3]:
# Find last year with good coverage
last_yr = df.dropna(subset=['ln_gdp_pc', 'ln_co2_pc']).year.max()
print(f'Last year with data: {last_yr}')

cs_2005 = df.query('year == 2005').dropna(subset=['ln_gdp_pc', 'ln_co2_pc'])
cs_last = df.query('year == @last_yr').dropna(subset=['ln_gdp_pc', 'ln_co2_pc'])

print(f'Cross-section 2005: N = {len(cs_2005)}')
print(f'Cross-section {last_yr}: N = {len(cs_last)}')

Last year with data: 2023
Cross-section 2005: N = 185
Cross-section 2023: N = 186


In [4]:
# Model 1: OLS 2005
m1 = pf.feols('ln_co2_pc ~ ln_gdp_pc', data=cs_2005, vcov='HC1')

# Model 2: OLS last year
m2 = pf.feols('ln_co2_pc ~ ln_gdp_pc', data=cs_last, vcov='HC1')

pf.etable([m1, m2], model_heads=['OLS 2005', f'OLS {last_yr}'], head_order='h')

GT(_tbl_data=  level_0             level_1                        0                       1
0    coef           ln_gdp_pc    1.226*** <br> (0.075)   1.067*** <br> (0.063)
1    coef           Intercept  -10.945*** <br> (0.685)  -9.638*** <br> (0.612)
2   stats        Observations                      185                     186
3   stats           S.E. type                   hetero                  hetero
4   stats       R<sup>2</sup>                    0.629                   0.624
5   stats  Adj. R<sup>2</sup>                    0.627                   0.622, _body=<great_tables._gt_data.Body object at 0x0000023D4FB40CB0>, _boxhead=Boxhead([ColInfo(var='level_0', type=<ColInfoTypeEnum.row_group: 3>, column_label='level_0', column_align='center', column_width=None), ColInfo(var='level_1', type=<ColInfoTypeEnum.stub: 2>, column_label='level_1', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000023D4FB40E00>, _spanners=Spanners([SpannerInfo(spanner_id='OLS 2005', spanner_level=1, spanner_label='OLS 2005', spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='OLS 2023', spanner_level=1, spanner_label='OLS 2023', spanner_units=None, spanner_pattern=None, vars=['1'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _source_notes=['Significance levels: * p < 0.05, ** p < 0.01, *** p < 0.001. Format of coefficient cell:\nCoefficient \n (Std. Error)'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x0000023D4FB414F0>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), ta

## 2. First Difference models

We take first differences to remove time-invariant country characteristics.
All FD models include year dummies as an aggregate time trend.

In [5]:
# Sort and compute first differences
df = df.sort_values(['country', 'year'])
g = df.groupby('country')

df['d_ln_co2_pc'] = g['ln_co2_pc'].diff()
df['d_ln_gdp_pc'] = g['ln_gdp_pc'].diff()
df['d_urban_pct'] = g['urban_pct'].diff()

# Lags of d_ln_gdp_pc (for FD with lags)
for i in range(1, 7):
    df[f'd_ln_gdp_pc_L{i}'] = g['d_ln_gdp_pc'].shift(i)

# Also need lags of d_urban_pct for confounder model 4
for i in range(1, 3):
    df[f'd_urban_pct_L{i}'] = g['d_urban_pct'].shift(i)

print(f'First differences computed. Non-null d_ln_co2_pc: {df.d_ln_co2_pc.notna().sum()}')

First differences computed. Non-null d_ln_co2_pc: 5828


In [6]:
# Model 3: FD, time trend, no lags
m3 = pf.feols('d_ln_co2_pc ~ d_ln_gdp_pc + C(year)',
              data=df, vcov={'CRV1': 'country'})

# Model 4: FD, time trend, 2-year lags
m4 = pf.feols('d_ln_co2_pc ~ d_ln_gdp_pc + d_ln_gdp_pc_L1 + d_ln_gdp_pc_L2 + C(year)',
              data=df, vcov={'CRV1': 'country'})

# Model 5: FD, time trend, 6-year lags
lag_vars = ' + '.join([f'd_ln_gdp_pc_L{i}' for i in range(1, 7)])
m5 = pf.feols(f'd_ln_co2_pc ~ d_ln_gdp_pc + {lag_vars} + C(year)',
              data=df, vcov={'CRV1': 'country'})

pf.etable([m3, m4, m5],
          model_heads=['FD no lags', 'FD 2 lags', 'FD 6 lags'],
          head_order='h',
          drop='year')

GT(_tbl_data=   level_0             level_1                      0                      1  \
0     coef         d_ln_gdp_pc  0.431*** <br> (0.058)  0.387*** <br> (0.061)   
1     coef      d_ln_gdp_pc_L1                            0.013 <br> (0.052)   
2     coef      d_ln_gdp_pc_L2                            0.050 <br> (0.030)   
3     coef      d_ln_gdp_pc_L3                                                 
4     coef      d_ln_gdp_pc_L4                                                 
5     coef      d_ln_gdp_pc_L5                                                 
6     coef      d_ln_gdp_pc_L6                                                 
7     coef           Intercept    -0.006 <br> (0.009)     0.010 <br> (0.009)   
8    stats        Observations                   5763                   5387   
9    stats           S.E. type            by: country            by: country   
10   stats       R<sup>2</sup>                  0.055                  0.054   
11   stats  Adj. R<sup>2</sup>                  0.050                  0.049   

                        2  
0   0.408*** <br> (0.069)  
1      0.042 <br> (0.060)  
2     -0.004 <br> (0.037)  
3      0.049 <br> (0.043)  
4      0.067 <br> (0.055)  
5     -0.044 <br> (0.052)  
6      0.064 <br> (0.042)  
7     -0.005 <br> (0.008)  
8                    4635  
9             by: country  
10                  0.059  
11                  0.052  , _body=<great_tables._gt_data.Body object at 0x0000023D54DFD970>, _boxhead=Boxhead([ColInfo(var='level_0', type=<ColInfoTypeEnum.row_group: 3>, column_label='level_0', column_align='center', column_width=None), ColInfo(var='level_1', type=<ColInfoTypeEnum.stub: 2>, column_label='level_1', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000023D54DFC920>, _spanners=Spanners([SpannerInfo(spanner_id='FD no lags', spanner_level=1, spanner_label='FD no lags', spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='FD 2 lags', spanner_level=1, spanner_label='FD 2 lags', spanner_units=None, spanner_pattern=None, vars=['1'], built=None), SpannerInfo(spanner_id='FD 6 lags', spanner_level=1, spanner_label='FD 6 lags', spanner_units=None, spanner_pattern=None, vars=['2'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _source_notes=['Significance levels: * p < 0.05, ** p < 0.01, *** p < 0.001. Format of coefficient cell:\nCoefficient \n (Std. Error)'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x0000023D54DFDF10>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table

### Cumulative (long-run) effects for FD models

The individual lag coefficients show how GDP changes propagate over time.
The cumulative effect (sum of contemporaneous + all lags) gives the **total long-run elasticity**.

In [7]:
# Cumulative effects
for name, model, n_lags in [('M3 (no lags)', m3, 0), ('M4 (2 lags)', m4, 2), ('M5 (6 lags)', m5, 6)]:
    coefs = model.coef()
    gdp_vars = ['d_ln_gdp_pc'] + [f'd_ln_gdp_pc_L{i}' for i in range(1, n_lags + 1)]
    cumul = sum(coefs[v] for v in gdp_vars if v in coefs.index)
    print(f'{name}: cumulative effect = {cumul:.4f}')

M3 (no lags): cumulative effect = 0.4311
M4 (2 lags): cumulative effect = 0.4510
M5 (6 lags): cumulative effect = 0.5841


## 3. Fixed Effects model

In [8]:
# Model 6: FE with country and year fixed effects
m6 = pf.feols('ln_co2_pc ~ ln_gdp_pc + C(year) | country',
              data=df, vcov={'CRV1': 'country'})

pf.etable([m6], model_heads=['FE'], head_order='h', drop='year')

GT(_tbl_data=  level_0               level_1                      0
0    coef             ln_gdp_pc  0.609*** <br> (0.079)
1      fe               country                      x
2   stats          Observations                   5951
3   stats             S.E. type            by: country
4   stats         R<sup>2</sup>                  0.966
5   stats  R<sup>2</sup> Within                  0.226, _body=<great_tables._gt_data.Body object at 0x0000023D56FB38F0>, _boxhead=Boxhead([ColInfo(var='level_0', type=<ColInfoTypeEnum.row_group: 3>, column_label='level_0', column_align='center', column_width=None), ColInfo(var='level_1', type=<ColInfoTypeEnum.stub: 2>, column_label='level_1', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000023D56FB0E00>, _spanners=Spanners([SpannerInfo(spanner_id='FE', spanner_level=1, spanner_label='FE', spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _source_notes=['Significance levels: * p < 0.05, ** p < 0.01, *** p < 0.001. Format of coefficient cell:\nCoefficient \n (Std. Error)'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x0000023D56FB05F0>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='hidden'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='ta

## 4. Summary table: all 6 models

In [9]:
# Display summary table
tbl_main = pf.etable([m1, m2, m3, m4, m5, m6],
          model_heads=['OLS 2005', f'OLS {last_yr}', 'FD', 'FD 2 lags', 'FD 6 lags', 'FE'],
          head_order='h',
          drop=r'year|C\(year\)',
          show_se_type=False,
          labels={
              'ln_gdp_pc': 'ln(GDP pc)',
              'd_ln_gdp_pc': '$\\Delta$ ln(GDP pc)',
              'd_ln_gdp_pc_L1': '$\\Delta$ ln(GDP pc)$_{t-1}$',
              'd_ln_gdp_pc_L2': '$\\Delta$ ln(GDP pc)$_{t-2}$',
              'd_ln_gdp_pc_L3': '$\\Delta$ ln(GDP pc)$_{t-3}$',
              'd_ln_gdp_pc_L4': '$\\Delta$ ln(GDP pc)$_{t-4}$',
              'd_ln_gdp_pc_L5': '$\\Delta$ ln(GDP pc)$_{t-5}$',
              'd_ln_gdp_pc_L6': '$\\Delta$ ln(GDP pc)$_{t-6}$',
              'Intercept': 'Constant',
          })
display(tbl_main)

# Export LaTeX
tex_main = pf.etable([m1, m2, m3, m4, m5, m6],
          model_heads=['OLS 2005', f'OLS {last_yr}', 'FD', 'FD 2 lags', 'FD 6 lags', 'FE'],
          head_order='h',
          drop=r'year|C\(year\)',
          show_se_type=False,
          labels={
              'ln_gdp_pc': 'ln(GDP pc)',
              'd_ln_gdp_pc': '$\\Delta$ ln(GDP pc)',
              'd_ln_gdp_pc_L1': '$\\Delta$ ln(GDP pc)$_{t-1}$',
              'd_ln_gdp_pc_L2': '$\\Delta$ ln(GDP pc)$_{t-2}$',
              'd_ln_gdp_pc_L3': '$\\Delta$ ln(GDP pc)$_{t-3}$',
              'd_ln_gdp_pc_L4': '$\\Delta$ ln(GDP pc)$_{t-4}$',
              'd_ln_gdp_pc_L5': '$\\Delta$ ln(GDP pc)$_{t-5}$',
              'd_ln_gdp_pc_L6': '$\\Delta$ ln(GDP pc)$_{t-6}$',
              'Intercept': 'Constant',
          },
          type='tex')
with open(f'{OUT}/tab_main.tex', 'w') as f:
    f.write(tex_main)
print(f'Saved {OUT}/tab_main.tex')

GT(_tbl_data=   level_0                      level_1                        0  \
0     coef                   ln(GDP pc)    1.226*** <br> (0.075)   
1     coef          $\Delta$ ln(GDP pc)                            
2     coef  $\Delta$ ln(GDP pc)$_{t-1}$                            
3     coef  $\Delta$ ln(GDP pc)$_{t-2}$                            
4     coef  $\Delta$ ln(GDP pc)$_{t-3}$                            
5     coef  $\Delta$ ln(GDP pc)$_{t-4}$                            
6     coef  $\Delta$ ln(GDP pc)$_{t-5}$                            
7     coef  $\Delta$ ln(GDP pc)$_{t-6}$                            
8     coef                     Constant  -10.945*** <br> (0.685)   
9       fe                      country                        -   
10   stats                 Observations                      185   
11   stats                R<sup>2</sup>                    0.629   
12   stats         R<sup>2</sup> Within                        -   

                         1                      2                      3  \
0    1.067*** <br> (0.063)                                                 
1                           0.431*** <br> (0.058)  0.387*** <br> (0.061)   
2                                                     0.013 <br> (0.052)   
3                                                     0.050 <br> (0.030)   
4                                                                          
5                                                                          
6                                                                          
7                                                                          
8   -9.638*** <br> (0.612)    -0.006 <br> (0.009)     0.010 <br> (0.009)   
9                        -                      -                      -   
10                     186                   5763                   5387   
11                   0.624                  0.055                  0.054   
12                       -                      -                      -   

                        4                      5  
0                          0.609*** <br> (0.079)  
1   0.408*** <br> (0.069)                         
2      0.042 <br> (0.060)                         
3     -0.004 <br> (0.037)                         
4      0.049 <br> (0.043)                         
5      0.067 <br> (0.055)                         
6     -0.044 <br> (0.052)                         
7      0.064 <br> (0.042)                         
8     -0.005 <br> (0.008)                         
9                       -                      x  
10                   4635                   5951  
11                  0.059                  0.966  
12                      -                  0.226  , _body=<great_tables._gt_data.Body object at 0x0000023D570B9CD0>, _boxhead=Boxhead([ColInfo(var='level_0', type=<ColInfoTypeEnum.row_group: 3>, column_label='level_0', column_align='center', column_width=None), ColInfo(var='level_1', type=<ColInfoTypeEnum.stub: 2>, column_label='level_1', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label='(4)', column_align='center', column_width=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label='(5)', column_align='center', column_width=None), ColInfo(var='5', type=<ColInfoTypeEnum.default: 1>, column_label='(6)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000023D54D89970>, _spanners=Spanners([SpannerInfo(spanner_id='OLS 2005', spanner_level=1, spanner_label='OLS 2005', spanner_units=None, spanner_pattern=None, vars=['0'], built=None), 

Saved ../output/tab_main.tex


## 5. Adding the confounder: Urbanization

Urbanization (% urban population) is a potential confounder: it drives both GDP growth
(through industrialization) and CO2 emissions (through energy-intensive urban infrastructure).
We add it to models 1 (OLS 2005), 4 (FD 2 lags), and 6 (FE).

In [10]:
# Model 1c: OLS 2005 + urbanization
m1c = pf.feols('ln_co2_pc ~ ln_gdp_pc + urban_pct', data=cs_2005, vcov='HC1')

# Model 4c: FD 2 lags + urbanization (also differenced, with lags)
m4c = pf.feols('d_ln_co2_pc ~ d_ln_gdp_pc + d_ln_gdp_pc_L1 + d_ln_gdp_pc_L2 '
               '+ d_urban_pct + d_urban_pct_L1 + d_urban_pct_L2 + C(year)',
               data=df, vcov={'CRV1': 'country'})

# Model 6c: FE + urbanization
m6c = pf.feols('ln_co2_pc ~ ln_gdp_pc + urban_pct + C(year) | country',
               data=df, vcov={'CRV1': 'country'})

conf_labels = {
    'ln_gdp_pc': 'ln(GDP pc)',
    'd_ln_gdp_pc': '$\\Delta$ ln(GDP pc)',
    'd_ln_gdp_pc_L1': '$\\Delta$ ln(GDP pc)$_{t-1}$',
    'd_ln_gdp_pc_L2': '$\\Delta$ ln(GDP pc)$_{t-2}$',
    'urban_pct': 'Urban pop. (\\%)',
    'd_urban_pct': '$\\Delta$ Urban (\\%)',
    'd_urban_pct_L1': '$\\Delta$ Urban (\\%)$_{t-1}$',
    'd_urban_pct_L2': '$\\Delta$ Urban (\\%)$_{t-2}$',
    'Intercept': 'Constant',
}
conf_heads = ['OLS 2005', 'OLS 2005 + conf.', 'FD 2 lags', 'FD 2 lags + conf.', 'FE', 'FE + conf.']

tbl_conf = pf.etable([m1, m1c, m4, m4c, m6, m6c],
          model_heads=conf_heads, head_order='h',
          drop=r'year|C\(year\)', show_se_type=False, labels=conf_labels)
display(tbl_conf)

# Export LaTeX
tex_conf = pf.etable([m1, m1c, m4, m4c, m6, m6c],
          model_heads=conf_heads, head_order='h',
          drop=r'year|C\(year\)', show_se_type=False, labels=conf_labels,
          type='tex')
with open(f'{OUT}/tab_confounder.tex', 'w') as f:
    f.write(tex_conf)
print(f'Saved {OUT}/tab_confounder.tex')

GT(_tbl_data=   level_0                      level_1                        0  \
0     coef                   ln(GDP pc)    1.226*** <br> (0.075)   
1     coef              Urban pop. (\%)                            
2     coef          $\Delta$ ln(GDP pc)                            
3     coef  $\Delta$ ln(GDP pc)$_{t-1}$                            
4     coef  $\Delta$ ln(GDP pc)$_{t-2}$                            
5     coef          $\Delta$ Urban (\%)                            
6     coef  $\Delta$ Urban (\%)$_{t-1}$                            
7     coef  $\Delta$ Urban (\%)$_{t-2}$                            
8     coef                     Constant  -10.945*** <br> (0.685)   
9       fe                      country                        -   
10   stats                 Observations                      185   
11   stats                R<sup>2</sup>                    0.629   
12   stats         R<sup>2</sup> Within                        -   

                          1                      2                      3  \
0     1.196*** <br> (0.122)                                                 
1        0.002 <br> (0.008)                                                 
2                            0.387*** <br> (0.061)  0.390*** <br> (0.060)   
3                               0.013 <br> (0.052)     0.013 <br> (0.052)   
4                               0.050 <br> (0.030)     0.050 <br> (0.030)   
5                                                     -0.007 <br> (0.005)   
6                                                      0.003 <br> (0.005)   
7                                                    0.016** <br> (0.005)   
8   -10.770*** <br> (0.852)     0.010 <br> (0.009)     0.007 <br> (0.009)   
9                         -                      -                      -   
10                      185                   5387                   5387   
11                    0.630                  0.054                  0.057   
12                        -                      -                      -   

                        4                      5  
0   0.609*** <br> (0.079)  0.584*** <br> (0.076)  
1                          0.016*** <br> (0.005)  
2                                                 
3                                                 
4                                                 
5                                                 
6                                                 
7                                                 
8                                                 
9                       x                      x  
10                   5951                   5951  
11                  0.966                  0.966  
12                  0.226                  0.247  , _body=<great_tables._gt_data.Body object at 0x0000023D56FBC4A0>, _boxhead=Boxhead([ColInfo(var='level_0', type=<ColInfoTypeEnum.row_group: 3>, column_label='level_0', column_align='center', column_width=None), ColInfo(var='level_1', type=<ColInfoTypeEnum.stub: 2>, column_label='level_1', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label='(4)', column_align='center', column_width=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label='(5)', column_align='center', column_width=None), ColInfo(var='5', type=<ColInfoTypeEnum.default: 1>, column_label='(6)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000023D57015C10>, _spanners=Spanners([SpannerInfo(spanner_id='OLS 2005', spanner_level=1, spanner_label='OLS 2005', spanner_units=None, spanner_pattern=None, vars=['0'],

Saved ../output/tab_confounder.tex
